# 🤖 04. Selenium — 동적 사이트 크롤링
## "BeautifulSoup으로 안 되는 것, Selenium으로 해결한다"

> **이전에 배운 것**: requests + BeautifulSoup → 서버가 완성해서 보내주는 HTML 수집
> **이번에 배울 것**: Selenium → JS가 만든 데이터, 버튼 클릭, 무한스크롤까지 수집

---

**이 노트북의 흐름:**
1. 왜 Selenium이 필요한가 (직접 눈으로 확인)
2. 설치 & import 해부
3. 브라우저 열기 & 옵션 설명
4. 요소 찾기 & 데이터 가져오기
5. Explicit Wait (가장 중요!)
6. 실전 ① JS 렌더링 사이트
7. 실전 ② 무한스크롤
8. 실전 ③ 사람인 버튼 클릭

---
# 1. 왜 Selenium이 필요한가?

## BeautifulSoup이 실패하는 순간

```
📌 웹사이트가 데이터를 보여주는 방식 2가지

방식 1. 정적 HTML (Static)
─────────────────────────────────────────
서버 → "여기 완성된 HTML 줄게"
        <div class="item">아이폰 15</div>  ← 이미 있음!

→ requests로 받은 HTML에 데이터가 있음 → BeautifulSoup OK ✅


방식 2. JavaScript 렌더링 (Dynamic)
─────────────────────────────────────────
서버 → "여기 빈 HTML 줄게"
        <div id="product-list">  <!-- 비어있음! -->
        </div>

브라우저 → "JavaScript 실행!"
        <div id="product-list">
          <div class="item">아이폰 15</div>  ← JS가 채워넣음
        </div>

→ requests는 JS 실행 못함 → 빈 HTML만 받음 → BeautifulSoup 실패 ❌
→ Selenium은 실제 Chrome을 켜서 JS까지 실행! ✅
```

---

## 30초 안에 확인하는 법 — Ctrl+U

```
1. 크롬에서 수집하고 싶은 페이지 열기
2. Ctrl + U  →  새 탭에 "페이지 소스"가 열림
3. Ctrl + F  →  수집하고 싶은 텍스트 검색

결과 있음 → 정적 HTML → BeautifulSoup으로 충분! ✅
결과 없음 → JS 렌더링 → Selenium 필요! 🤖
```

**지금 직접 해봅시다:**
- `http://quotes.toscrape.com` 열기 → Ctrl+U → "Tolstoy" 검색 → **있음** (정적)
- `http://quotes.toscrape.com/js` 열기 → Ctrl+U → "Tolstoy" 검색 → **없음!** (JS 렌더링)

---
## 📌 먼저 BeautifulSoup의 실패를 직접 눈으로 확인해봅시다

같은 사이트인데 URL 끝에 `/js`만 붙어있어요.
BeautifulSoup으로 긁으면 어떤 결과가 나올까요?

In [ ]:
# ❌ BeautifulSoup으로 JS 렌더링 사이트 시도
# — 이게 왜 안 되는지 직접 확인!

import requests
from bs4 import BeautifulSoup

# 정적 버전 (BeautifulSoup으로 OK)
response_static = requests.get('http://quotes.toscrape.com/')
soup_static = BeautifulSoup(response_static.text, 'html.parser')
quotes_static = soup_static.select('.quote')
print(f"정적 버전: {len(quotes_static)}개 발견 ✅")

print()

# JS 렌더링 버전 (BeautifulSoup으로 실패)
response_js = requests.get('http://quotes.toscrape.com/js/')
soup_js = BeautifulSoup(response_js.text, 'html.parser')
quotes_js = soup_js.select('.quote')
print(f"JS 렌더링 버전: {len(quotes_js)}개 발견 ← 0개! JS가 아직 실행 안 됨")
print()
print("→ 이때 Selenium이 필요합니다!")


---
# 2. 설치

터미널(VS Code 하단 터미널)에서 실행:
```bash
pip install selenium webdriver-manager
```

**`webdriver-manager`를 쓰는 이유:**
```
❌ 예전 방식: Chrome 버전 확인 → ChromeDriver 사이트에서 직접 다운로드 → 경로 설정
              Chrome 업데이트할 때마다 반복... 번거로움!

✅ webdriver-manager: 자동으로 내 Chrome 버전에 맞는 드라이버 설치!
              ChromeDriverManager().install() 한 줄로 끝!
```

In [ ]:
# 설치 확인

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'show', 'selenium', 'webdriver-manager'],
    capture_output=True, text=True
)
for line in result.stdout.split('\n'):
    if line.startswith('Name') or line.startswith('Version'):
        print(line)

print("\n패키지 확인 완료! 위에 Name/Version이 보이면 정상이에요.")


---
# 3. import — "이 줄들이 다 뭐야?"

아래 6줄이 처음 보면 압도적으로 느껴져요.
한 줄씩 비유로 설명할게요.

```python
from selenium import webdriver
# → "selenium 도구함에서 '브라우저 조종기'를 꺼냄"
#   driver = webdriver.Chrome(...) 할 때 쓰는 바로 그것!

from selenium.webdriver.chrome.service import Service
# → "크롬 드라이버를 실행하는 '서비스 관리자'를 꺼냄"
#   Chrome을 안정적으로 시작하고 종료하는 역할

from selenium.webdriver.common.by import By
# → "요소를 찾는 방법을 지정하는 '주소 형식'을 꺼냄"
#   By.CSS_SELECTOR, By.ID 같은 것들

from selenium.webdriver.support.ui import WebDriverWait
# → "요소가 나타날 때까지 기다려주는 '타이머'를 꺼냄"
#   최대 N초 동안 조건이 만족될 때까지 대기

from selenium.webdriver.support import expected_conditions as EC
# → "기다릴 조건들의 모음집"
#   as EC = 이름이 너무 길어서 EC로 줄임
#   EC.presence_of_element_located 같은 것들

from webdriver_manager.chrome import ChromeDriverManager
# → "ChromeDriver 자동 설치기"
```

> 💡 **외울 필요 없어요!**
> "Selenium 쓸 때 항상 맨 위에 붙이는 주문" 정도로만 기억하세요.
> 실무에서도 다들 복붙해서 씁니다.

In [ ]:
# Selenium import
# ─────────────────────────────────────────────────────
# 아래 6줄은 Selenium 쓸 때 항상 맨 위에 붙이는 "주문"이에요.
# 외우려 하지 말고, 복붙해서 쓰면 됩니다!
# ─────────────────────────────────────────────────────

from selenium import webdriver                                    # 브라우저 조종기
from selenium.webdriver.chrome.service import Service             # 크롬 실행 관리자
from selenium.webdriver.common.by import By                       # 요소 찾는 방법 지정
from selenium.webdriver.support.ui import WebDriverWait           # 대기 타이머
from selenium.webdriver.support import expected_conditions as EC  # 대기 조건 모음 (EC로 줄임)
from webdriver_manager.chrome import ChromeDriverManager          # ChromeDriver 자동 설치
from selenium.webdriver.common.keys import Keys                   # 키보드 특수키 (엔터 등)
import time       # 대기용
import pandas as pd  # 결과를 DataFrame으로

print("✅ import 완료!")
print("다음 셀을 실행하면 Chrome이 자동으로 열립니다.")


---
# 4. Chrome 브라우저 열기 — 옵션이 뭔지?

```
options.add_argument(...)은 Chrome을 켜기 전에 "설정값을 미리 지정"하는 것이에요.
앱을 켜기 전에 설정 → 환경설정 하는 것처럼요.

'--disable-gpu'
→ 그래픽 카드(GPU) 사용 안 함
   코드로 브라우저를 띄울 때 GPU 관련 오류가 자주 나서 끄는 것
   크롤링에는 화면 렌더링 품질이 필요 없으니까요

'--no-sandbox'
→ 보안 샌드박스 해제
   특정 환경(Linux, 일부 Windows)에서 오류 방지용
   일반적인 크롤링에서는 보안 문제 없어요

'--window-size=1280,900'
→ 브라우저 창 크기를 1280x900 픽셀로 고정
   화면 크기에 따라 보이는 요소가 달라질 수 있어서 (모바일 vs PC)

'user-agent=Mozilla/5.0...'
→ "나는 일반 사람이 쓰는 Chrome이에요"라고 사이트에 알리기
   Selenium으로 접속하면 봇으로 감지되는 경우가 있음
   이 옵션을 붙이면 일반 사용자처럼 보임
```

> ✅ 이 옵션 블록 전체를 통째로 복붙해서 쓰면 됩니다.
> AI에게 "Selenium 옵션 세팅해줘" 해도 비슷한 코드가 나와요.

In [ ]:
# Chrome 브라우저 열기
# ─────────────────────────────────────────────────────
# 이 셀을 실행하면 Chrome 창이 하나 열려요!
# ─────────────────────────────────────────────────────

# 옵션 설정 (Chrome 켜기 전 환경설정)
options = webdriver.ChromeOptions()
# options.add_argument('--headless')  # ← 창 없이 실행 (서버 배포용, 지금은 주석처리)
options.add_argument('--disable-gpu')           # GPU 오류 방지
options.add_argument('--no-sandbox')            # 샌드박스 오류 방지
options.add_argument('--window-size=1280,900')  # 창 크기 고정
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0')
# user-agent: "나는 일반 Chrome 사용자예요"라고 사이트에 알리기

# 드라이버 실행 (Chrome 실제로 켜기)
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),  # ChromeDriver 자동 설치 & 실행
    options=options
)

print("✅ Chrome이 열렸습니다!")
print(f"Chrome 버전: {driver.capabilities['browserVersion']}")
print()
print("💡 driver = 브라우저를 조종하는 '리모컨'이에요.")
print("   driver.get(URL)  → URL로 이동")
print("   driver.quit()    → 브라우저 종료 (마지막에 꼭 해주세요!)")


---
# 5. 기본 브라우저 제어

```python
driver.get('URL')       # URL로 이동
driver.title            # 현재 페이지 제목
driver.current_url      # 현재 URL
driver.back()           # 뒤로 가기
driver.forward()        # 앞으로 가기
driver.refresh()        # 새로고침
driver.page_source      # 현재 페이지 HTML (JS 실행 후!)
driver.quit()           # 브라우저 종료 (항상 마지막에!)
```

In [ ]:
# 기본 내비게이션 실습

# 페이지 이동
driver.get('http://quotes.toscrape.com/')
time.sleep(2)  # 페이지 완전히 로드될 때까지 잠깐 대기

print(f"제목: {driver.title}")
print(f"URL: {driver.current_url}")

# page_source = Selenium이 가져온 HTML (JS 실행 후!)
# BeautifulSoup의 response.text와 다른 점: JS 실행 결과가 포함됨!
page_source = driver.page_source
print(f"\n소스 크기: {len(page_source):,} 글자")
print("소스 미리보기 (처음 200자):")
print(page_source[:200])


---
# 6. 요소 찾기 — find_element vs find_elements

```
💡 BeautifulSoup과 비교하면?

BeautifulSoup                  Selenium
────────────────────────────────────────────
soup.select_one('.quote')  →  driver.find_element(By.CSS_SELECTOR, '.quote')
soup.select('.quote')      →  driver.find_elements(By.CSS_SELECTOR, '.quote')

개념이 똑같아요! 이름만 달라진 것.
```

| 함수 | 반환 | 없을 때 |
|------|------|---------|
| `find_element` | 요소 하나 | 💥 에러 발생 |
| `find_elements` | 요소 리스트 | `[]` 빈 리스트 |

```
By 종류 (CSS_SELECTOR 하나만 알면 99% 해결!):

By.CSS_SELECTOR    '.class', '#id', 'div.card'  ← 가장 많이 씀 ✅
By.XPATH           //div[@class='name']          ← 복잡한 경우
By.ID              'search-box'                  ← id 속성
By.TAG_NAME        'div', 'a', 'span'            ← 태그 이름
```

In [ ]:
# 요소 찾기 실습
# 현재 quotes.toscrape.com에 있어야 해요!

# ─── find_element: 하나만 (첫 번째 요소)
first_quote = driver.find_element(By.CSS_SELECTOR, '.quote')
print("첫 번째 명언 블록 전체 텍스트 (앞 100자):")
print(first_quote.text[:100])

print()

# ─── find_elements: 전부 다
all_quotes = driver.find_elements(By.CSS_SELECTOR, '.quote')
print(f"총 명언 수: {len(all_quotes)}개")

print()

# ─── 요소 안에서 다시 찾기 (중첩 선택)
# quote 블록 하나 안에 .text, .author, .tag 가 있어요
print("── 처음 3개 명언 ──")
for i, quote in enumerate(all_quotes[:3], 1):
    text = quote.find_element(By.CSS_SELECTOR, '.text').text   # 블록 안에서 찾기!
    author = quote.find_element(By.CSS_SELECTOR, '.author').text
    print(f"\n{i}. {author}")
    print(f"   {text[:60]}...")


---
# 7. 텍스트 & 속성값 가져오기

```python
element.text                       # 화면에 보이는 텍스트
element.get_attribute('href')      # 링크 주소
element.get_attribute('src')       # 이미지 주소
element.get_attribute('class')     # 클래스명
element.get_attribute('data-id')   # data 속성
element.get_attribute('value')     # input 박스의 입력값
element.get_attribute('innerHTML') # 내부 HTML 전체
```

> 💡 BeautifulSoup에서 `tag['href']` 했던 것 = Selenium에서 `get_attribute('href')`

In [ ]:
# 텍스트와 속성값 가져오기

first_quote = driver.find_element(By.CSS_SELECTOR, '.quote')

# 텍스트
text_el = first_quote.find_element(By.CSS_SELECTOR, '.text')
print(f"[텍스트] {text_el.text[:60]}")

# a 태그의 href (링크 주소)
author_link = first_quote.find_element(By.CSS_SELECTOR, 'a')
print(f"[href]  {author_link.get_attribute('href')}")

# 클래스명 확인
span_el = first_quote.find_element(By.CSS_SELECTOR, '.text')
print(f"[class] {span_el.get_attribute('class')}")

# 태그 리스트 (여러 개)
tags = first_quote.find_elements(By.CSS_SELECTOR, '.tag')
tag_texts = [t.text for t in tags]  # 리스트 컴프리헨션으로 한 줄에 정리
print(f"[태그들] {tag_texts}")


---
# 8. 클릭 & 텍스트 입력

```python
# 기본 클릭
element.click()

# JS 클릭 (일반 클릭이 안 될 때 — 팝업 등이 가리고 있는 경우)
driver.execute_script("arguments[0].click();", element)

# 텍스트 입력
element.clear()           # 기존 내용 지우기
element.send_keys('텍스트')  # 입력

# 특수키
element.send_keys(Keys.ENTER)    # 엔터
element.send_keys(Keys.TAB)      # 탭
element.send_keys(Keys.ESCAPE)   # ESC
```

In [ ]:
# 클릭 실습: '다음 페이지' 버튼 클릭

print(f"현재 페이지: {driver.current_url}")

# '다음' 버튼 찾기
next_btn = driver.find_element(By.CSS_SELECTOR, 'li.next a')
print(f"다음 버튼 링크: {next_btn.get_attribute('href')}")

# 실제로 클릭!
next_btn.click()
time.sleep(2)  # 페이지 로드 대기

print(f"\n이동 후 페이지: {driver.current_url}")
print(f"제목: {driver.title}")
print("\n→ URL이 /page/2/ 로 바뀌었죠? 버튼 클릭이 성공했어요! ✅")


---
# 9. ⭐ Explicit Wait — 가장 중요한 개념!

## Selenium에서 제일 자주 보는 에러:
```
NoSuchElementException: 요소를 찾지 못했습니다
```

## 원인:
```
페이지 이동 → JS 실행 시작 → (0.5초 후) 데이터 채워짐
               ↑
           여기서 바로 find_element 하면 → 아직 없음 → 에러!
```

## time.sleep vs WebDriverWait 비교:

```
time.sleep(3):
→ "무조건 3초 기다림"
   음식이 1초에 나와도 3초 기다림 (낭비)
   음식이 5초 걸리면 3초 후에 에러 (위험!)

WebDriverWait + EC:
→ "요소가 나타날 때까지 기다리되, 최대 10초"
   1초에 나오면 1초만 기다림 (효율적)
   10초 지나도 안 나오면 그때 에러 (안전)
```

## 코드 해석:
```python
wait = WebDriverWait(driver, 10)
#                    ↑       ↑
#               브라우저  최대 대기 시간(초)

element = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, '.quote'))
)
# wait.until(조건)  = "이 조건이 만족될 때까지 기다려라"
# EC.presence_of_element_located = "이 요소가 DOM에 나타나면"
# (By.CSS_SELECTOR, '.quote')    = "CSS .quote인 요소"
#
# 전체 의미: "'.quote' 요소가 DOM에 나타날 때까지 최대 10초 기다려라"
```

In [ ]:
# Explicit Wait 실습

driver.get('http://quotes.toscrape.com/')
# time.sleep 없이 바로 Wait으로!

# 대기 객체 생성 (최대 10초)
wait = WebDriverWait(driver, 10)

# '.quote' 요소가 나타날 때까지 대기
quotes = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote'))
)

print(f"✅ 대기 완료! {len(quotes)}개 명언 발견")
print()
print("💡 자주 쓰는 EC 조건:")
print("  presence_of_all_elements_located → 여러 요소가 DOM에 있을 때 (리스트 수집 전)")
print("  presence_of_element_located      → 요소 하나가 DOM에 있을 때")
print("  element_to_be_clickable          → 버튼을 클릭할 수 있을 때")
print("  visibility_of_element_located    → 요소가 화면에 보일 때 (팝업 등)")


---
# 10. 스크롤 제어 — execute_script

## execute_script가 뭔지?

```
비유: 파이썬은 한국어, 브라우저는 JavaScript(영어)만 알아들어요.
      Selenium이 통역사 역할을 해줍니다.

driver.execute_script("JS 코드")
→ "브라우저에게 JS 명령을 보내는 것"

JS를 몰라도 됩니다!
스크롤 관련은 아래 3가지만 기억하세요:
```

```python
# 맨 아래로 스크롤
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

# 맨 위로 스크롤
driver.execute_script("window.scrollTo(0, 0);")

# 조금 아래로 (500px)
driver.execute_script("window.scrollBy(0, 500);")
```

> ✅ 이 3줄 중 하나를 골라 복붙하면 됩니다. 나머지는 AI에게 시키면 돼요.

In [ ]:
# 스크롤 실습

driver.get('http://quotes.toscrape.com/')
time.sleep(2)

# 현재 스크롤 위치
pos_before = driver.execute_script("return window.pageYOffset")
print(f"스크롤 전 위치: {pos_before}px")

# 500px 아래로 스크롤
driver.execute_script("window.scrollBy(0, 500);")
time.sleep(1)

pos_after = driver.execute_script("return window.pageYOffset")
print(f"스크롤 후 위치: {pos_after}px")

# 페이지 전체 높이
total_height = driver.execute_script("return document.body.scrollHeight")
print(f"페이지 전체 높이: {total_height}px")

# 맨 아래로
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(1)

pos_bottom = driver.execute_script("return window.pageYOffset")
print(f"맨 아래 위치: {pos_bottom}px")


---
# 11. 실전 ① — JS 렌더링 사이트 크롤링

`http://quotes.toscrape.com/js/`

URL 끝에 `/js/`만 붙었는데 — 이게 JavaScript 렌더링 버전이에요.

아까 셀 1에서 BeautifulSoup으로 시도했을 때 **0개**가 나왔던 그 사이트!
Selenium으로는 가져올 수 있어요.

In [ ]:
# ❌ BeautifulSoup으로 시도 (0개가 나오는 것 다시 확인)
import requests
from bs4 import BeautifulSoup

response = requests.get('http://quotes.toscrape.com/js/')
soup = BeautifulSoup(response.text, 'html.parser')
quotes_bs = soup.select('.quote')
print(f"BeautifulSoup: {len(quotes_bs)}개 발견")
print("← JS가 아직 실행 안 됐기 때문에 비어있어요!")


In [ ]:
# ✅ Selenium으로 JS 렌더링 사이트 크롤링
# — 실제 Chrome을 켜서 JS가 실행된 결과를 가져옴!

driver.get('http://quotes.toscrape.com/js/')

# JS 실행이 완료될 때까지 대기 (중요!)
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')))

quotes_el = driver.find_elements(By.CSS_SELECTOR, '.quote')
print(f"Selenium: {len(quotes_el)}개 발견 ✅")
print()

# 데이터 추출
rows = []
for q in quotes_el:
    text = q.find_element(By.CSS_SELECTOR, '.text').text
    author = q.find_element(By.CSS_SELECTOR, '.author').text
    tags_el = q.find_elements(By.CSS_SELECTOR, '.tag')
    tags = ', '.join(t.text for t in tags_el)  # 태그 리스트를 문자열로

    rows.append({'명언': text, '저자': author, '태그': tags})

df = pd.DataFrame(rows)
print(df[['저자', '명언']].to_string(index=False))


In [ ]:
# 여러 페이지 수집 — 버튼 클릭으로 페이지 이동
# (URL을 직접 바꾸는 게 아니라, "다음" 버튼을 클릭하는 방식!)

driver.get('http://quotes.toscrape.com/js/')

all_quotes = []
page = 1

while page <= 5:  # 최대 5페이지
    # 현재 페이지의 명언이 로드될 때까지 대기
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')))

    quotes_el = driver.find_elements(By.CSS_SELECTOR, '.quote')

    # 현재 페이지 데이터 수집
    for q in quotes_el:
        text = q.find_element(By.CSS_SELECTOR, '.text').text
        author = q.find_element(By.CSS_SELECTOR, '.author').text
        all_quotes.append({'페이지': page, '명언': text, '저자': author})

    print(f"  {page}페이지: {len(quotes_el)}개 수집 (누적 {len(all_quotes)}개)")

    # 다음 페이지 버튼 있는지 확인
    next_btns = driver.find_elements(By.CSS_SELECTOR, 'li.next a')
    if not next_btns:            # 다음 버튼이 없으면 → 마지막 페이지!
        print("  마지막 페이지 도달!")
        break

    next_btns[0].click()        # 다음 버튼 클릭
    page += 1
    time.sleep(1)

df_quotes = pd.DataFrame(all_quotes)
print(f"\n총 {len(df_quotes)}개 명언 수집 완료!")
print(df_quotes.groupby('저자').size().sort_values(ascending=False).head(5))


---
# 12. 실전 ② — 무한스크롤 크롤링

무한스크롤: 페이지 끝까지 내리면 새 데이터가 자동으로 로드!
(인스타그램, 트위터, 유튜브 스타일)

**전략:**
```
1. 현재 아이템 수 기록
2. 스크롤을 맨 아래로 내림
3. 새 데이터 로드될 때까지 대기 (time.sleep)
4. 아이템 수가 늘었으면 → 새 데이터 수집
5. 아이템 수가 그대로면 → 끝! break
```

In [ ]:
# 무한스크롤 크롤링: quotes.toscrape.com/scroll

driver.get('http://quotes.toscrape.com/scroll')
time.sleep(2)  # 첫 로드 대기

all_quotes = []
last_count = 0     # 이전에 수집한 아이템 수
scroll_count = 0
max_scrolls = 15   # 최대 15번 스크롤

print("무한스크롤 시작!")

while scroll_count < max_scrolls:
    # 현재 화면에 있는 명언 전부 수집
    quotes_el = driver.find_elements(By.CSS_SELECTOR, '.quote')
    current_count = len(quotes_el)

    # 새 데이터가 없으면 (스크롤해도 안 늘어나면) 종료
    if current_count == last_count:
        print(f"  더 이상 새 데이터 없음. 종료!")
        break

    # 새로 추가된 것만 저장 (last_count 이후부터)
    for q in quotes_el[last_count:]:
        text = q.find_element(By.CSS_SELECTOR, '.text').text
        author = q.find_element(By.CSS_SELECTOR, '.author').text
        all_quotes.append({'명언': text, '저자': author})

    print(f"  스크롤 {scroll_count + 1}: 누적 {current_count}개 (새 {current_count - last_count}개 추가)")
    last_count = current_count  # 수 업데이트
    scroll_count += 1

    # 맨 아래로 스크롤 (새 데이터 로드 트리거)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # 데이터 로드 기다리기

df_scroll = pd.DataFrame(all_quotes)
print(f"\n총 {len(df_scroll)}개 명언 수집 완료!")
df_scroll.head(5)


---
# 13. 실전 ③ — 사람인 채용공고 수집

사람인에서 '데이터분석' 키워드로 검색한 공고를 수집해봐요!

Selenium을 쓰는 이유:
- 팝업이 자동으로 뜰 수 있음 → 닫아야 함
- 필터 버튼 클릭 후 결과 수집 가능
- 로그인 후 더 많은 정보 접근 가능

> ⚠️ **주의**: 사이트 구조가 바뀌면 CSS 선택자가 달라질 수 있어요.
> 에러가 나면 개발자도구(F12) → Inspector에서 선택자를 다시 확인하세요.

In [ ]:
# 사람인 Selenium — 채용공고 수집

driver.get('https://www.saramin.co.kr/zf_user/search/recruit?searchword=데이터분석')
time.sleep(3)  # 페이지 로드 대기

# ── 팝업 처리 ──────────────────────────────────────────────
# 사람인은 팝업이 뜰 수 있어요.
# try-except: "팝업이 있으면 닫고, 없으면 그냥 넘어가"
try:
    close_btns = driver.find_elements(
        By.CSS_SELECTOR, '.btn-close, .close-btn, #layerRecmd .close'
    )
    for btn in close_btns:
        if btn.is_displayed():  # 실제로 화면에 보이는지 확인
            btn.click()
            time.sleep(0.5)
            print("팝업 닫음")
except:
    print("팝업 없음 (정상)")

# ── 채용공고 로드 대기 & 수집 ──────────────────────────────
wait = WebDriverWait(driver, 10)

try:
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.item_recruit')))
    jobs_el = driver.find_elements(By.CSS_SELECTOR, 'div.item_recruit')
    print(f"발견된 공고: {len(jobs_el)}개")
except:
    print("⚠️ 공고를 찾지 못했어요. 사이트 구조가 바뀌었을 수 있어요.")
    print("   F12 → Inspector에서 공고 박스의 CSS 선택자를 다시 확인해보세요.")
    jobs_el = []

# ── 데이터 추출 ────────────────────────────────────────────
rows = []
for job in jobs_el:
    try:
        company = job.find_element(By.CSS_SELECTOR, 'strong.corp_name').text.strip()
        title = job.find_element(By.CSS_SELECTOR, 'h2.job_tit').text.strip()

        # 조건들 (지역, 경력 등)
        conditions = job.find_elements(By.CSS_SELECTOR, 'div.job_condition span')
        conds = [c.text.strip() for c in conditions]

        rows.append({
            '회사명': company,
            '공고제목': title,
            '지역': conds[0] if conds else None,
            '경력': conds[1] if len(conds) > 1 else None,
        })
    except Exception:
        continue  # 파싱 실패한 공고는 건너뜀

df_saramin = pd.DataFrame(rows)
print(f"\n수집된 공고 {len(df_saramin)}개:")
print(df_saramin[['회사명', '공고제목', '경력']].head(10).to_string(index=False))


---
# 14. 에러 해결 가이드

| 에러 | 원인 | 해결 |
|------|------|------|
| `NoSuchElementException` | 요소를 못 찾음 | CSS 선택자 확인, WebDriverWait 추가 |
| `ElementClickInterceptedException` | 팝업 등이 위에 있음 | 팝업 먼저 닫기, JS 클릭 사용 |
| `StaleElementReferenceException` | 페이지가 바뀌면서 요소 사라짐 | 요소를 다시 find_element로 찾기 |
| `TimeoutException` | Wait 시간 초과 | 대기 시간 늘리기, 선택자 확인 |
| 브라우저 창이 안 열림 | ChromeDriver 버전 문제 | `pip install --upgrade webdriver-manager` |

**JS 클릭 (일반 클릭이 안 될 때):**
```python
# 팝업이나 다른 요소가 가리고 있어서 클릭이 안 될 때
driver.execute_script("arguments[0].click();", element)
```

In [ ]:
# JS 실행 실습 — 스크롤 & 클릭

driver.get('http://quotes.toscrape.com/')
time.sleep(2)

# 페이지 높이
height = driver.execute_script("return document.body.scrollHeight")
print(f"페이지 높이: {height}px")

# 맨 아래로
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(1)

pos = driver.execute_script("return window.pageYOffset")
print(f"스크롤 후 위치: {pos}px")

# 특정 요소로 스크롤
footer = driver.find_element(By.CSS_SELECTOR, 'footer')
driver.execute_script("arguments[0].scrollIntoView(true);", footer)
print("\n푸터로 스크롤 완료!")


In [ ]:
# ⚠️ 브라우저 종료 — 항상 마지막에!
#
# driver.quit() 없이 끝내면:
# → Chrome 프로세스가 백그라운드에 계속 살아있음 (메모리 낭비)
# → 다음에 driver.get() 하면 에러

driver.quit()
print("✅ 브라우저 종료 완료!")


---
# ✅ 전체 코드 템플릿 — 실무에서 바로 쓰는 형태

```python
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, pandas as pd

# ── 1. 브라우저 열기
options = webdriver.ChromeOptions()
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--window-size=1280,900')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
wait = WebDriverWait(driver, 10)

try:
    # ── 2. 페이지 이동
    driver.get('크롤링할 URL')
    time.sleep(1)

    # ── 3. 팝업 처리 (있으면)
    try:
        close_btn = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, '.popup-close')
        ))
        close_btn.click()
    except:
        pass  # 팝업 없으면 그냥 넘어감

    # ── 4. 데이터 수집
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.item')))
    items = driver.find_elements(By.CSS_SELECTOR, '.item')

    rows = []
    for item in items:
        name = item.find_element(By.CSS_SELECTOR, '.name').text
        rows.append({'이름': name})
        time.sleep(0.2)  # 서버에 부담 주지 않기

    # ── 5. 저장
    df = pd.DataFrame(rows)
    df.to_csv('결과.csv', index=False, encoding='utf-8-sig')
    print(f'✅ 완료: {len(df)}행 저장')

finally:
    driver.quit()  # ← 항상 종료! try-finally로 보장
```

---

## BeautifulSoup vs Selenium 언제 써야 하나?

| 상황 | 도구 |
|------|------|
| HTML 소스(Ctrl+U)에 데이터가 있음 | BeautifulSoup ✅ (빠름) |
| HTML 소스에 데이터가 없음 (JS 렌더링) | Selenium |
| 스크롤해야 데이터가 로드됨 | Selenium |
| 버튼 클릭 / 필터 / 로그인 필요 | Selenium |
| 수천 페이지 대량 수집 | BeautifulSoup (속도 이점) |

---

## 🎯 핵심 메시지

> **Selenium 코드가 길고 복잡해 보이는 이유:**
> 실제 Chrome을 열고, 조종하고, 기다리고, 닫는 모든 과정을 코드로 써야 해서요.
>
> 하지만 **패턴은 딱 하나**예요:
> "열기 → 이동 → 대기 → 수집 → 닫기"
> 이 틀 안에서 수집 부분만 바꾸는 거예요.
>
> **코드를 외우지 마세요.**
> 템플릿을 복붙하고, AI에게 "이 사이트에서 이 데이터 Selenium으로 수집해줘"라고 시키면 됩니다.
> 여러분은 그 코드를 읽고 "아, 이 선택자가 이 요소를 찾는 거구나"를 판단할 수 있으면 충분합니다.

---
# ✋ 연습문제

### 문제 1 (기초)
`http://quotes.toscrape.com/js/`에서 명언, 저자, 태그를 수집하여
DataFrame으로 만들고 CSV로 저장하세요.

(힌트: BeautifulSoup으로는 0개가 나와요! Selenium 필요 🤖)

### 문제 2 (응용)
`http://quotes.toscrape.com/scroll`에서 무한스크롤로
모든 명언을 수집하세요.

(힌트: 총 몇 개나 나오는지 확인해보세요!)

### 문제 3 (실전)
사람인에서 '데이터' 키워드로 검색한 뒤,
2페이지까지 다음 버튼을 클릭하며 공고를 수집하세요.